# Librerias

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import re
import causalidad as cs

# Lectura de datos

In [ ]:
df_icare = pd.read_excel(r"C:\Users\afpue\OneDrive\Documentos\GitHub\causalidad\Codigo\df_icare.xlsx")

# Funciones

## Calcular el ATE

In [ ]:
cols = ['actfreq_sq001', 'recgov_sq001']

df_icare[cols] = df_icare[cols].apply(lambda x: (x == 1).astype(int))
resultado = cs.calcular_ate(
    df=df_icare,
    resultado='actfreq_sq001',
    tratamiento='recgov_sq001',
    covariables=['sex', 'age', 'edu']
)

if resultado is not None:
    print(f"ATE (diferencia de medias): {resultado['ate_diff']:.4f}")
    print(f"ATE (regresión ajustada):   {resultado['ate_reg']:.4f} (p-valor: {resultado['p_valor_reg']:.4f})")
    print(f"N tratados: {resultado['n_tratados']}, N controles: {resultado['n_controles']}")

ATE (diferencia de medias): 0.4244
ATE (regresión ajustada):   0.4119 (p-valor: 0.0000)
N tratados: 1645, N controles: 18


In [ ]:
import re

# 1. Identificar todos los sufijos disponibles buscando columnas 'actfreq_sqXXX'
sufijos = [re.search(r'sq(\d+)', col).group(1) 
           for col in df_icare.columns 
           if col.startswith('actfreq_sq')]

resultados_totales = []

# 2. Iterar sobre cada sufijo para procesar los pares
for num in sorted(sufijos):
    col_resultado = f'actfreq_sq{num}'
    col_tratamiento = f'recgov_sq{num}'
    
    # Verificar si el par existe en el dataset
    if col_tratamiento in df_icare.columns:
        print(f"\n--- Procesando Par: {col_resultado} y {col_tratamiento} ---")
        
        # Limpieza: Convertir a 0/1 (siguiendo tu lógica de x == 1)
        # Es vital asegurar que el tratamiento sea binario para el modelo logit
        df_par = df_icare.copy()
        df_par[[col_resultado, col_tratamiento]] = df_par[[col_resultado, col_tratamiento]].apply(
            lambda x: (x == 1).astype(int)
        )
        
        # Llamada a tu función de ATE
        res = cs.calcular_ate(
            df=df_par,
            resultado=col_resultado,
            tratamiento=col_tratamiento,
            covariables=['sex', 'age', 'edu']
        )
        
        if res is not None:
            print(f"ATE (diferencia): {res['ate_diff']:.4f}")
            print(f"ATE (regresión):  {res['ate_reg']:.4f} (p-v: {res['p_valor_reg']:.4f})")
            resultados_totales.append({'par': num, 'ate': res['ate_diff']})
        else:
            print(f"No se pudo calcular el ATE para el par {num} (posible falta de varianza o datos).")

NameError: name 'df_icare' is not defined

## Balancear

In [ ]:
# Para matching
df_match = cs.balancear_propensity(df_icare, 'recgov_sq001', 'actfreq_sq001', 
                                     ['sex','age','edu'], metodo='matched', replacement=False)

ate_match = cs.calcular_ate(df_match, 'actfreq_sq001', 'recgov_sq001', covariables=['sex','age','edu'])

if ate_match is not None:
    print(f"ATE (diferencia de medias): {ate_match['ate_diff']:.4f}")
    print(f"ATE (regresión ajustada):   {ate_match['ate_reg']:.4f} (p-valor: {ate_match['p_valor_reg']:.4f})")
    print(f"N tratados: {ate_match['n_tratados']}, N controles: {ate_match['n_controles']}")

# Para subclassification
df_sub = cs.balancear_propensity(df_icare, 'recgov_sq001', 'actfreq_sq001', 
                                   ['sex','age','edu'], metodo='subclassification', n_subclases=5)

ate_sub = cs.calcular_ate(df_sub, 'actfreq_sq001', 'recgov_sq001', covariables=['sex','age','edu'])

if ate_sub is not None:
    print(f"ATE (diferencia de medias): {ate_sub['ate_diff']:.4f}")
    print(f"ATE (regresión ajustada):   {ate_sub['ate_reg']:.4f} (p-valor: {ate_sub['p_valor_reg']:.4f})")
    print(f"N tratados: {ate_sub['n_tratados']}, N controles: {ate_sub['n_controles']}")

ATE (diferencia de medias): 0.4444
ATE (regresión ajustada):   0.4253 (p-valor: 0.0062)
N tratados: 18, N controles: 18
ATE (diferencia de medias): 0.4244
ATE (regresión ajustada):   0.4119 (p-valor: 0.0000)
N tratados: 1645, N controles: 18
